In [2]:
import tensorflow as tf 
from tensorflow import keras 
import tensorflow_addons as tfa 
import pandas as pd
import numpy as np 
from models import load_mf_model
#import gensim.downloader as api
#from utils import load_annotations, load_transcriptions, process_text, preprocess_text, loss_val_graph

c:\Users\SIA\anaconda3\envs\deepface-env\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


### Load data

In [3]:
AUTOTUNE = tf.data.AUTOTUNE

# Train
scene_train_ds = tf.data.experimental.load('./data/fullscene/train_ds/')
face_train_ds  = tf.data.experimental.load('./data/faces/train_ds/')
audio_train_ds = tf.data.experimental.load('./data/audio/train_ds/')
text_train_ds  = tf.data.experimental.load('./data/text/train_ds/').batch(batch_size=32)

scene_xtrain = scene_train_ds.map(lambda x,y: x)
face_xtrain  = face_train_ds.map(lambda x,y: x)
audio_xtrain = audio_train_ds.map(lambda x,y: x)
text__xtrain = text_train_ds.map(lambda x,y: x)
y_train      = scene_train_ds.map(lambda x,y: y)

train_ds = tf.data.Dataset.zip(((scene_xtrain, face_xtrain, audio_xtrain, text__xtrain), y_train)).shuffle(buffer_size=1000).prefetch(buffer_size=AUTOTUNE)


# Valid
scene_valid_ds = tf.data.experimental.load('./data/fullscene/val_ds/')
face_valid_ds  = tf.data.experimental.load('./data/faces/val_ds/')
audio_valid_ds = tf.data.experimental.load('./data/audio/val_ds') 
text_valid_ds  = tf.data.experimental.load('./data/text/val_ds/').batch(batch_size=32)

scene_xvalid = scene_valid_ds.map(lambda x,y: x)
face_xvalid  = face_valid_ds.map(lambda x,y: x)
audio_xvalid = audio_valid_ds.map(lambda x,y: x)
text_xvalid  = text_valid_ds.map(lambda x,y: x)
y_valid      = scene_valid_ds.map(lambda x,y: y)

valid_ds = tf.data.Dataset.zip(((scene_xvalid, face_xvalid, audio_xvalid, text_xvalid), y_valid)).shuffle(buffer_size=1000).prefetch(buffer_size=AUTOTUNE)

train_ds, valid_ds

Instructions for updating:
Use `tf.data.Dataset.load(...)` instead.


(<_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None, 50), dtype=tf.int32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>,
 <_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None, 50), dtype=tf.int32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>)

### Load model

In [4]:
mf_model = load_mf_model()
mf_model.save_weights('./weights/mf/mf_model.t5')


### Load weights

In [5]:
mf_model.load_weights('./weights/mf/mf_model.t5')

In [6]:
loss, mae = mf_model.evaluate(valid_ds)
(1-mae)*100

1/1 [==============================] - 14s 14s/step - loss: 0.0177 - mae: 0.1043


89.57254439592361

### Load data

In [7]:
scene_test_ds = tf.data.experimental.load('./data/fullscene/test_ds/')
face_test_ds  = tf.data.experimental.load('./data/faces/test_ds/')
audio_test_ds = tf.data.experimental.load('./data/audio/test_ds') 
text_test_ds  = tf.data.experimental.load('./data/text/test_ds/').batch(batch_size=32)


scene_xtest = scene_test_ds.map(lambda x,y: x)
face_xtest  = face_test_ds.map(lambda x,y: x)
audio_xtest = audio_test_ds.map(lambda x,y: x)
text_xtest  = text_test_ds.map(lambda x,y: x)

y_test      = scene_test_ds.map(lambda x,y: y)

test_ds = tf.data.Dataset.zip(((scene_xtest, face_xtest, audio_xtest, text_xtest), y_test)).shuffle(buffer_size=1000).prefetch(buffer_size=AUTOTUNE)

test_ds

<_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None, 50), dtype=tf.int32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>

In [8]:
loss, mae = mf_model.evaluate(test_ds)
(1-mae)*100

1/1 [==============================] - 4s 4s/step - loss: 0.0050 - mae: 0.0615


93.85163299739361

In [9]:
train_ds, valid_ds

(<_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None, 50), dtype=tf.int32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>,
 <_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None, 50), dtype=tf.int32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>)

In [10]:
import datetime
t = datetime.datetime.now().strftime("%m%d_%H%M%S")

early_stopping = keras.callbacks.EarlyStopping(patience=10, verbose=0)
check_point    = keras.callbacks.ModelCheckpoint(filepath='./weights/mf/mf.t5',
                             monitor='val_mae',
                             mode='min',
                             save_best_only=True,
                             save_weights_only=True,
                             verbose=0)

optimizer = tfa.optimizers.RectifiedAdam()
mf_model.compile(loss='mse', optimizer=optimizer, metrics=['mae'])

### Retrain model

In [11]:

history = mf_model.fit(train_ds, validation_data=valid_ds, batch_size=8, epochs=100, callbacks=[early_stopping, check_point])

Epoch 1/100
1/1 [==============================] - 46s 46s/step - loss: 0.0380 - mae: 0.1409 - val_loss: 0.0177 - val_mae: 0.1042
Epoch 2/100
1/1 [==============================] - 34s 34s/step - loss: 0.0430 - mae: 0.1719 - val_loss: 0.0177 - val_mae: 0.1042
Epoch 3/100
1/1 [==============================] - 36s 36s/step - loss: 0.0414 - mae: 0.1539 - val_loss: 0.0176 - val_mae: 0.1041
Epoch 4/100
1/1 [==============================] - 26s 26s/step - loss: 0.0496 - mae: 0.1923 - val_loss: 0.0176 - val_mae: 0.1040
Epoch 5/100
1/1 [==============================] - 42s 42s/step - loss: 0.0353 - mae: 0.1556 - val_loss: 0.0176 - val_mae: 0.1040
Epoch 6/100
1/1 [==============================] - 29s 29s/step - loss: 0.0576 - mae: 0.2047 - val_loss: 0.0174 - val_mae: 0.1034
Epoch 7/100
1/1 [==============================] - 38s 38s/step - loss: 0.0461 - mae: 0.1770 - val_loss: 0.0172 - val_mae: 0.1026
Epoch 8/100
1/1 [==============================] - 37s 37s/step - loss: 0.0378 - mae: 0.16

### Load weights

In [12]:
mf_model.load_weights('./weights/mf/mf.t5')

## Evaluation

### Validation data

In [13]:
AUTOTUNE = tf.data.AUTOTUNE
scene_valid_ds = tf.data.experimental.load('./data/fullscene/val_ds/')
face_valid_ds  = tf.data.experimental.load('./data/faces/val_ds/')
audio_valid_ds = tf.data.experimental.load('./data/audio/val_ds') 
text_valid_ds  = tf.data.experimental.load('./data/text/val_ds/').batch(batch_size=32)

scene_xvalid = scene_valid_ds.map(lambda x,y: x)
face_xvalid  = face_valid_ds.map(lambda x,y: x)
audio_xvalid = audio_valid_ds.map(lambda x,y: x)
text_xvalid  = text_valid_ds.map(lambda x,y: x)
y_valid      = scene_valid_ds.map(lambda x,y: y)

valid_ds = tf.data.Dataset.zip(((scene_xvalid, face_xvalid, audio_xvalid, text_xvalid), y_valid)).prefetch(buffer_size=AUTOTUNE)


In [14]:
loss, mae = mf_model.evaluate(valid_ds)

1/1 [==============================] - 10s 10s/step - loss: 0.0014 - mae: 0.0293


In [15]:
from sklearn.metrics import mean_absolute_error 

y_true = np.concatenate([y for x,y in valid_ds], axis=0)
y_pred = mf_model.predict(valid_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

1/1 [==============================] - 13s 13s/step


(array([97.818054, 99.41916 , 97.957924, 97.47411 , 92.667404],
       dtype=float32),
 97.06732984632254)

### Test data

In [16]:
scene_test_ds = tf.data.experimental.load('./data/fullscene/test_ds/')
face_test_ds  = tf.data.experimental.load('./data/faces/test_ds/')
audio_test_ds = tf.data.experimental.load('./data/audio/test_ds') 
text_test_ds  = tf.data.experimental.load('./data/text/test_ds/').batch(batch_size=32)


scene_xtest = scene_test_ds.map(lambda x,y: x)
face_xtest  = face_test_ds.map(lambda x,y: x)
audio_xtest = audio_test_ds.map(lambda x,y: x)
text_xtest  = text_test_ds.map(lambda x,y: x)

y_test      = scene_test_ds.map(lambda x,y: y)

test_ds = tf.data.Dataset.zip(((scene_xtest, face_xtest, audio_xtest, text_xtest), y_test)).prefetch(buffer_size=AUTOTUNE) #.shuffle(buffer_size=1000)

test_ds

<_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None, 50), dtype=tf.int32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>

In [17]:
y_true = np.concatenate([y for x,y in test_ds], axis=0)
y_pred = mf_model.predict(test_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

1/1 [==============================] - 9s 9s/step


(array([91.90355 , 87.514114, 95.60174 , 97.08832 , 95.44275 ],
       dtype=float32),
 93.51009353995323)

In [18]:
(1-mae)*100

array([91.90355 , 87.514114, 95.60174 , 97.08832 , 95.44275 ],
      dtype=float32)

In [20]:
import os
import pickle

os.makedirs("./histories", exist_ok=True)

with open('./histories/mf.pkl', 'wb') as f:
    pickle.dump(history.history, f)
